In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_pseudobulk_de"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_pseudobulk_de"

# ----------------------------
# M1/M2 polarization scoring — direct extension of Azizi et al. 2018's
# finding that M1/M2 marker genes are co-expressed in the same cells,
# tested here at single-cell resolution using our own validated
# macrophage sub-clusters (now confirmed correct after today's fixes).
# Also checks where the tumour-vs-normal reprogramming genes (FN1,
# HSPA1A/B) sit relative to this axis.
# ----------------------------
adata1_mac = sc.read_h5ad(PROCESSED_DIR / "GSE114725_macrophages_subclustered.h5ad")
print(f"Loaded: {adata1_mac.n_obs} macrophages")

m1_genes = ["IL1B", "TNF", "NOS2", "CD86", "CXCL9", "CXCL10"]
m2_genes = ["CD163", "MRC1", "IL10", "ARG1", "CD206", "MSR1"]

mac_raw = adata1_mac.raw.to_adata()
mac_raw.obs["mac_subtype"] = adata1_mac.obs["mac_subtype"].values

m1_present = [g for g in m1_genes if g in mac_raw.var_names]
m2_present = [g for g in m2_genes if g in mac_raw.var_names]
print(f"M1 markers found: {m1_present}")
print(f"M2 markers found: {m2_present}")

sc.tl.score_genes(mac_raw, m1_present, score_name="M1_score")
sc.tl.score_genes(mac_raw, m2_present, score_name="M2_score")

r, p = pearsonr(mac_raw.obs["M1_score"], mac_raw.obs["M2_score"])
print(f"\nM1 vs M2 score correlation across all macrophages: r={r:.3f}, p={p:.2e}")

# Per sub-cluster breakdown
print("\nM1/M2 scores by sub-cluster:")
print(mac_raw.obs.groupby("mac_subtype", observed=True)[["M1_score", "M2_score"]].mean().round(3))

# Scatter plot, coloured by sub-cluster
fig, ax = plt.subplots(figsize=(8, 7))
for subtype in mac_raw.obs["mac_subtype"].unique():
    mask = mac_raw.obs["mac_subtype"] == subtype
    ax.scatter(mac_raw.obs.loc[mask, "M1_score"], mac_raw.obs.loc[mask, "M2_score"],
               label=subtype, alpha=0.5, s=10)
ax.set_xlabel("M1 score")
ax.set_ylabel("M2 score")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title(f"M1 vs M2 polarization score, per cell (r={r:.2f})")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_macrophage_M1_M2_spectrum.png", dpi=300, bbox_inches="tight")
plt.close()

# Where do the reprogramming genes sit relative to this axis?
print("\nReprogramming genes (FN1, HSPA1A/B) vs M1/M2 axis:")
for gene in ["FN1", "HSPA1A", "HSPA1B"]:
    if gene in mac_raw.var_names:
        X = mac_raw[:, gene].X
        if hasattr(X, "toarray"): X = X.toarray()
        r_m1, _ = pearsonr(X.flatten(), mac_raw.obs["M1_score"])
        r_m2, _ = pearsonr(X.flatten(), mac_raw.obs["M2_score"])
        print(f"  {gene}: correlation with M1={r_m1:.3f}, M2={r_m2:.3f}")

print("\nSaved: GSE114725_macrophage_M1_M2_spectrum.png")

Loaded: 5914 macrophages
M1 markers found: ['IL1B', 'TNF', 'CD86', 'CXCL9', 'CXCL10']
M2 markers found: ['CD163', 'MRC1', 'IL10', 'MSR1']

M1 vs M2 score correlation across all macrophages: r=0.007, p=5.95e-01

M1/M2 scores by sub-cluster:
                                                    M1_score  M2_score
mac_subtype                                                           
LAM-like macrophages                                  -0.027    -0.174
Lipid-laden/Foam-cell macrophages                     -0.108    -0.182
Antigen-presenting macrophages                         0.083    -0.158
Complement-high macrophages                            0.044    -0.071
Resting/Resident macrophages                          -0.044     0.571
Monocyte-like macrophages                              0.062    -0.167
Non-classical monocytes (CD16+)                       -0.025    -0.538
Unassigned (n=91, stromal/RBC contamination art...    -0.110    -0.258

Reprogramming genes (FN1, HSPA1A/B) vs M1/M2 axis

In [2]:
import json
from pathlib import Path

NOTEBOOKS_DIR = Path(r"C:\Users\annam\Dissertation 2026\Notebooks")
for nb_path in NOTEBOOKS_DIR.glob("*.ipynb"):
    nb = json.load(open(nb_path, encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        src = "".join(cell.get("source", []))
        if "_raw.h5ad" in src and "clean_rawcounts" not in src:
            print(f"{nb_path.name}, cell {i}: loads _raw.h5ad directly")

01_GSE114725_data_loading.ipynb, cell 6: loads _raw.h5ad directly
01_GSE114725_data_loading.ipynb, cell 8: loads _raw.h5ad directly
01_GSE114725_data_loading.ipynb, cell 9: loads _raw.h5ad directly
01_GSE114725_data_loading.ipynb, cell 11: loads _raw.h5ad directly
02_GSE176078_data_loading.ipynb, cell 8: loads _raw.h5ad directly
02_GSE176078_data_loading.ipynb, cell 9: loads _raw.h5ad directly
03_phase1_QC_V2.ipynb, cell 2: loads _raw.h5ad directly
03_phase1_QC_V2.ipynb, cell 3: loads _raw.h5ad directly
10_phase4_QC_sensitivity_GSE176078.ipynb, cell 2: loads _raw.h5ad directly
11_phase4_normalisation_sensitivity_GSE176078.ipynb, cell 2: loads _raw.h5ad directly
12_phase4_GSE114725_sensitivity.ipynb, cell 2: loads _raw.h5ad directly


In [3]:
import json
from pathlib import Path

NOTEBOOKS_DIR = Path(r"C:\Users\annam\Dissertation 2026\Notebooks")
for nb_name in ["10_phase4_QC_sensitivity_GSE176078.ipynb",
                "11_phase4_normalisation_sensitivity_GSE176078.ipynb",
                "12_phase4_GSE114725_sensitivity.ipynb"]:
    nb = json.load(open(NOTEBOOKS_DIR / nb_name, encoding="utf-8"))
    full_text = "".join("".join(c.get("source", [])) for c in nb["cells"])
    has_vdj_filter = "vdj_mask" in full_text or "vdj_prefixes" in full_text
    has_mt_filter = 'startswith("MT-")' in full_text
    print(f"{nb_name}: VDJ filter present={has_vdj_filter}, MT filter present={has_mt_filter}")

10_phase4_QC_sensitivity_GSE176078.ipynb: VDJ filter present=True, MT filter present=True
11_phase4_normalisation_sensitivity_GSE176078.ipynb: VDJ filter present=True, MT filter present=True
12_phase4_GSE114725_sensitivity.ipynb: VDJ filter present=True, MT filter present=True
